# 6 - Hyperparameter Grid Search

**Covers:** the *Hyperparameter grid* subsection of Section 4.2.2. One configuration is selected per algorithm on the fixed 15% validation partition by minimising validation RMSE, then **frozen and applied unchanged to all models in that algorithm's ladder**.

| | grid | fixed, not searched |
|---|---|---|
| XGBoost | `max_depth` {4, 6, 8} x `learning_rate` {0.01, 0.05, 0.1} | `n_estimators` 1000 with 50-round early stopping on val RMSE; `subsample` = `colsample_bytree` = 0.8 |
| Random Forest | `n_estimators` {200, 500} x `min_samples_leaf` {1, 5} | `max_depth` unrestricted; `max_features` = floor(p/3) |

## One decision the paper leaves open

Section 4.2.2 says the configuration is selected by grid search but does not say **which feature configuration the search runs on**. Tuning on Alignment-Augmented would let the hyperparameters be chosen in the treatment's favour. This notebook therefore tunes on **Control**, the reference point of the central comparison, which is the conservative choice and cannot be accused of tilting the result.

`TUNE_ON` below makes that switchable. Expect the Data Scientist reviewer (Section 4.9.3 C) to ask about it -- the answer should be in the chapter.

**Runtime warning.** The Random Forest grid fits 4 forests with unrestricted depth on ~180k rows; the 500-tree settings are slow and memory-hungry. Set `N_JOBS` to suit the machine, and consider running this notebook's RF section on its own.

**Produces:** `artifacts/config_xgb.json`, `artifacts/config_rf.json`, `artifacts/tuning_results.csv`.

In [ ]:
import itertools
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

sys.path.insert(0, os.path.abspath('.'))
import modeling_config as mc

TUNE_ON = 'control'   # see the note above
N_JOBS = -1

df = mc.load_corpus()
splits = pd.read_csv(mc.artifact('splits.csv'))
df = df.merge(splits[['id', 'partition']], on='id', how='inner')
df = df[df['partition'].isin(['train', 'val', 'test'])].reset_index(drop=True)

X_all, blocks = mc.build_feature_frame(df)
mc.assert_clean(X_all)
y = df[mc.TARGET].astype(float)

cols = mc.columns_for(TUNE_ON, blocks)
is_train, is_val = df['partition'] == 'train', df['partition'] == 'val'
X_tr, y_tr = X_all.loc[is_train, cols], y[is_train]
X_va, y_va = X_all.loc[is_val, cols], y[is_val]

print('tuning on the %r feature configuration' % TUNE_ON)
print('train %s   val %s   features %d' % (X_tr.shape, X_va.shape, len(cols)))

## XGBoost - 9 combinations

In [ ]:
XGB_FIXED = dict(
    n_estimators=1000,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    eval_metric='rmse',
    early_stopping_rounds=50,
    random_state=mc.RANDOM_SEED,
    n_jobs=N_JOBS,
    tree_method='hist',
)

rows = []
for depth, lr in itertools.product([4, 6, 8], [0.01, 0.05, 0.1]):
    t0 = time.time()
    model = xgb.XGBRegressor(max_depth=depth, learning_rate=lr, **XGB_FIXED)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    val_rmse = float(np.sqrt(np.mean((y_va - model.predict(X_va)) ** 2)))
    rows.append({
        'algorithm': 'xgboost',
        'max_depth': depth,
        'learning_rate': lr,
        'best_iteration': int(model.best_iteration),
        'val_rmse': val_rmse,
        'seconds': round(time.time() - t0, 1),
    })
    print('depth=%d lr=%.2f -> val RMSE %.4f  (best_iter %d, %.1fs)'
          % (depth, lr, val_rmse, model.best_iteration, rows[-1]['seconds']))

xgb_results = pd.DataFrame(rows).sort_values('val_rmse')
print()
print(xgb_results.to_string(index=False))

In [ ]:
best_xgb = xgb_results.iloc[0]
config_xgb = {
    'algorithm': 'xgboost',
    'tuned_on_feature_config': TUNE_ON,
    'selection_metric': 'validation RMSE (fixed 15% partition)',
    'params': {
        'max_depth': int(best_xgb['max_depth']),
        'learning_rate': float(best_xgb['learning_rate']),
        'n_estimators': 1000,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'objective': 'reg:squarederror',
        'random_state': mc.RANDOM_SEED,
        'tree_method': 'hist',
    },
    'early_stopping_rounds': 50,
    'best_iteration_at_selection': int(best_xgb['best_iteration']),
    'val_rmse': float(best_xgb['val_rmse']),
}
print(json.dumps(config_xgb, indent=2))
print(mc.save_json(config_xgb, 'config_xgb.json'))

## Random Forest - 4 combinations

`max_features` is fixed at floor(p/3), the convention Breiman (2001) recommends for regression specifically, as distinct from classification's sqrt(p) rule. **This is the slow cell.**

In [ ]:
max_features = max(1, len(cols) // 3)
print('p = %d features -> max_features = floor(p/3) = %d' % (len(cols), max_features))

rows = []
for n_est, leaf in itertools.product([200, 500], [1, 5]):
    t0 = time.time()
    model = RandomForestRegressor(
        n_estimators=n_est,
        min_samples_leaf=leaf,
        max_depth=None,
        max_features=max_features,
        random_state=mc.RANDOM_SEED,
        n_jobs=N_JOBS,
    )
    model.fit(X_tr, y_tr)
    val_rmse = float(np.sqrt(np.mean((y_va - model.predict(X_va)) ** 2)))
    rows.append({
        'algorithm': 'random_forest',
        'n_estimators': n_est,
        'min_samples_leaf': leaf,
        'val_rmse': val_rmse,
        'seconds': round(time.time() - t0, 1),
    })
    print('n_estimators=%d min_samples_leaf=%d -> val RMSE %.4f  (%.1fs)'
          % (n_est, leaf, val_rmse, rows[-1]['seconds']))
    del model

rf_results = pd.DataFrame(rows).sort_values('val_rmse')
print()
print(rf_results.to_string(index=False))

In [ ]:
best_rf = rf_results.iloc[0]
config_rf = {
    'algorithm': 'random_forest',
    'tuned_on_feature_config': TUNE_ON,
    'selection_metric': 'validation RMSE (fixed 15% partition)',
    'params': {
        'n_estimators': int(best_rf['n_estimators']),
        'min_samples_leaf': int(best_rf['min_samples_leaf']),
        'max_depth': None,
        'max_features_rule': 'floor(p/3), recomputed per feature configuration',
        'random_state': mc.RANDOM_SEED,
    },
    'val_rmse': float(best_rf['val_rmse']),
}
print(json.dumps(config_rf, indent=2))
print(mc.save_json(config_rf, 'config_rf.json'))

all_results = pd.concat([xgb_results, rf_results], ignore_index=True)
path = mc.artifact('tuning_results.csv')
all_results.to_csv(path, index=False)
print(path)

---
### Notes for the chapter and for the Section 4.9.3(C) review

- Report the **full grid**, not just the winner. `tuning_results.csv` is that table.
- `max_features = floor(p/3)` is recomputed per feature configuration, because p differs across rungs of the ladder (Lyric-Only has p = 1). This is a property of Breiman's rule, not a per-model retune, so it does not violate the fixed-configuration requirement -- but say so explicitly, because it looks like a variation if left unexplained.
- The selected `best_iteration` is recorded but **not** frozen: early stopping runs again for each rung against the same validation partition, which is what Section 4.2.2 describes.

Next: `7_model_ladder.ipynb`.